# ZLS-Leidenfrost-3D-Paper-Soft -- Droplet Shape Evaluation

Compares the droplet interface (`Phi`, at the last timestep of the most recent successful
session in this project) against a stored reference shape.

Since the droplet is axisymmetric and the domain is 3D, only a thin slice near the `z = 0`
symmetry plane is compared (reduces to a 2D contour, same style as the 2D droplet-shape
checks elsewhere in this repo, e.g. `RisingBubble_Testcase1_Evaluation.ipynb`).

Run after `Leidenfrost_3d_HFSI_Fast_Paper_Soft.ipynb` has populated the project database with a session.

On first run (no reference file yet) this notebook saves the current shape as the new
reference and does **not** fail -- commit the generated `Leidenfrost_3d_HFSI_Fast_Paper_Soft_reference_shape.txt` file, then re-run
to actually validate future changes against it.


In [ ]:
#r "BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using System.IO;
using System.Data;
using System.Globalization;
using System.Threading;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.Gnuplot;
using BoSSS.Solution.XNSECommon;
using BoSSS.Application.BoSSSpad;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();


In [ ]:
BoSSSshell.WorkflowMgm.Init("ZLS-Leidenfrost-3D-Paper-Soft");


In [ ]:
var sess = BoSSSshell.WorkflowMgm.Sessions
    .Where(s => s.SuccessfulTermination)
    .OrderByDescending(s => s.CreationTime)
    .First();
Console.WriteLine("Using session: " + sess.Name + " (" + sess.ID + ")");


In [ ]:
var ts = sess.Timesteps.Last();
DGField phi = ts.GetField("Phi");
LevelSet LevSet = new LevelSet(phi.Basis, "LevelSet");
LevSet.Acc(1.0, phi);

var LsTrk = new LevelSetTracker((GridData)phi.GridDat, CutCellQuadratureMethod.Saye, 1, new string[] { "A", "B" }, LevSet);
LsTrk.UpdateTracker(ts.PhysicalTime);

MultidimensionalArray interP = XNSEUtils.GetInterfacePoints(LsTrk, LevSet, quadRuleOrderForNodeSet: 10);
int numP = interP.Lengths[0];
Console.WriteLine($"Extracted {numP} interface points at t = {ts.PhysicalTime}");


In [ ]:
double[] X = numP.ForLoop(i => interP[i, 0]);
double[] Y = numP.ForLoop(i => interP[i, 1]);
double[] Z = numP.ForLoop(i => interP[i, 2]);

// the domain uses a symmetry (freeslip) boundary at z = 0;
// take a thin slice near that plane so the 3D interface reduces to a 2D contour comparable
// to the 2D droplet-shape checks elsewhere in this repo (e.g. RisingBubble_Testcase1_Evaluation.ipynb)
double zTol = 0.02 * (Z.Max() - Z.Min());
var sliceIdx = Enumerable.Range(0, numP).Where(i => Math.Abs(Z[i]) <= zTol).ToArray();

// sort by polar angle around the slice centroid, so the points trace a closed contour
// (sorting by x alone zig-zags back and forth across a closed shape)
double xc = sliceIdx.Average(i => X[i]);
double yc = sliceIdx.Average(i => Y[i]);
sliceIdx = sliceIdx.OrderBy(i => Math.Atan2(Y[i] - yc, X[i] - xc)).ToArray();

double[] xSlice = sliceIdx.Select(i => X[i]).ToArray();
double[] ySlice = sliceIdx.Select(i => Y[i]).ToArray();
Console.WriteLine($"{xSlice.Length} points within {zTol:e2} of the z=0 symmetry plane");

var computedShape = new Plot2Ddata.XYvalues("computed", xSlice, ySlice);


In [ ]:
// same nearest-point average-distance check as RisingBubble_Testcase1_Evaluation.ipynb
public static void CheckInterfaceShapeAgainstReferenceData(Plot2Ddata.XYvalues xyValues, Plot2Ddata.XYvalues refValues, double tolerance) {

    double distError = 0.0;
    for (int i = 0; i < refValues.Values.Length; i++) {
        double minDist = double.MaxValue;
        for (int j = 0; j < xyValues.Values.Length; j++) {
            double dist = Math.Sqrt((refValues.Abscissas[i] - xyValues.Abscissas[j]).Pow2() + (refValues.Values[i] - xyValues.Values[j]).Pow2());
            if (dist < minDist)
                minDist = dist;
        }
        distError += minDist;
    }
    distError /= refValues.Values.Length;

    Console.WriteLine($"Interface shape distance error: {distError.ToString("E6", CultureInfo.InvariantCulture)} (tolerance: {tolerance.ToString("E6", CultureInfo.InvariantCulture)})");

    NUnit.Framework.Assert.IsTrue(distError <= tolerance,
        $"Absolute distance error {distError} of interface shape exceeds tolerance {tolerance}.");
}

In [ ]:
string refFileName = "Leidenfrost_3d_HFSI_Fast_Paper_Soft_reference_shape.txt";
double tolerance = 2e-3; // same order of magnitude as RisingBubble_Testcase1_Evaluation.ipynb; tighten once a few runs establish the expected scatter

var shapePlt = new Plot2Ddata();
var computedFmt = new PlotFormat { Style = Styles.Lines, LineColor = LineColors.Red, LineWidth = 2 };

// the simulated domain only covers X in [-L, 0] (droplet centered at X0 = 0), relying on the
// freeslip/symmetry boundary at x = 0; mirror across x = 0 here so the plot shows the full,
// physically symmetric droplet -- this is for the plot only, the error check above still
// compares the un-mirrored half-domain data
double[] xMirrored = xSlice.Concat(xSlice.Select(x => -x)).ToArray();
double[] yMirrored = ySlice.Concat(ySlice).ToArray();
double xcMirrored = xMirrored.Average();
double ycMirrored = yMirrored.Average();
var mirroredOrder = Enumerable.Range(0, xMirrored.Length)
    .OrderBy(i => Math.Atan2(yMirrored[i] - ycMirrored, xMirrored[i] - xcMirrored)).ToArray();
double[] xPlot = mirroredOrder.Select(i => xMirrored[i]).ToArray();
double[] yPlot = mirroredOrder.Select(i => yMirrored[i]).ToArray();

shapePlt.AddDataGroup("computed", xPlot, yPlot, computedFmt);

if (!File.Exists(refFileName)) {
    Console.WriteLine($"No reference shape found at '{refFileName}' -- saving the current (manually verified) result as the new reference.");
    Console.WriteLine("Commit this file to the repository so future runs are actually checked against it.");
    File.WriteAllLines(refFileName, xSlice.Zip(ySlice, (x, y) => $"{x.ToString(CultureInfo.InvariantCulture)} {y.ToString(CultureInfo.InvariantCulture)}"));
} else {
    string[] lines = File.ReadAllLines(refFileName);
    double[] xRef = new double[lines.Length];
    double[] yRef = new double[lines.Length];
    for (int i = 0; i < lines.Length; i++) {
        var parts = lines[i].Split(new[] { ' ' }, StringSplitOptions.RemoveEmptyEntries);
        xRef[i] = Convert.ToDouble(parts[0], CultureInfo.InvariantCulture);
        yRef[i] = Convert.ToDouble(parts[1], CultureInfo.InvariantCulture);
    }
    var referenceShape = new Plot2Ddata.XYvalues("reference", xRef, yRef);

    // same mirroring for the reference, plot only
    double[] xRefMirrored = xRef.Concat(xRef.Select(x => -x)).ToArray();
    double[] yRefMirrored = yRef.Concat(yRef).ToArray();

    var refFmt = new PlotFormat { Style = Styles.Points, PointType = PointTypes.Circle, LineColor = LineColors.Black };
    shapePlt.AddDataGroup("reference", xRefMirrored, yRefMirrored, refFmt);

    CheckInterfaceShapeAgainstReferenceData(computedShape, referenceShape, tolerance);
    Console.WriteLine("Droplet shape matches the reference within tolerance.");
}

shapePlt.Xlabel = "x";
shapePlt.Ylabel = "y";
shapePlt.AddGnuplotCommand("set size ratio -1"); // equal unit length on both axes, so the droplet shape isn't distorted
shapePlt.ToGnuplot().PlotSVG(xRes: 800, yRes: 800)

## 3D droplet shape via ParaView

Exports the terminal timestep and uses `pvpython` (ParaView's Python interpreter,
must be installed and on `PATH`) to render the `Phi = 0` isosurface -- the actual 3D
droplet shape -- and embeds the resulting screenshot. Same approach as
`internal/src/private-rckmnn/notebooks/SFB1194-K24-Y-Channel/`.

In [ ]:
string paraviewExportDir = Path.GetFullPath("./ParaViewLeidenfrost");
if (Directory.Exists(paraviewExportDir)) {
    Directory.Delete(paraviewExportDir, true);
}
sess.Timesteps.Last().Export().To(paraviewExportDir).WithSupersampling(3).Do(true);

In [ ]:
#!pwsh
pvpython ./leidenfrost_contour.py ./ParaViewLeidenfrost ./ParaViewLeidenfrost/leidenfrost_contour.png

In [ ]:
#!html
<img src="./ParaViewLeidenfrost/leidenfrost_contour.png" />